# 3 — Hyperparameter Sweep

**Goal:** Find the best learning rate, knowledge distillation temperature, and alpha before full training.

Target:
**Sweep** — 9 combinations × 5 epochs × 1,000 samples → find best LR, Temperature, Alpha

Next → src/train.py for full training on 25,380 samples

Run `1_setup_dataset.ipynb` first to extract the dataset.

In [ ]:
import os
import sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Clone repo + add src/ to path
PROJECT_DIR = '/content/project'
SRC_DIR     = os.path.join(PROJECT_DIR, 'src')

if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/Arjun11x/deepfake-audio-detection.git {PROJECT_DIR}

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Configure paths
import config
config.ENV          = "colab"
config.DATASET_ROOT = "/content/asvspoof2019/LA"
config.AUDIO_DIRS   = {
    "train" : "/content/asvspoof2019/LA/ASVspoof2019_LA_train/flac",
    "dev"   : "/content/asvspoof2019/LA/ASVspoof2019_LA_dev/flac",
    "eval"  : "/content/asvspoof2019/LA/ASVspoof2019_LA_eval/flac",
}
config.PROTOCOL_FILES = {
    "train" : "/content/asvspoof2019/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt",
    "dev"   : "/content/asvspoof2019/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt",
    "eval"  : "/content/asvspoof2019/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt",
}
config.SAVE_DIR = "/content/drive/MyDrive/deepfake_detector/models"
os.makedirs(config.SAVE_DIR, exist_ok=True)

# Extract dataset if needed
ZIP_PATH    = '/content/drive/MyDrive/LA.zip'
EXTRACT_DIR = '/content/asvspoof2019'
LA_DIR      = os.path.join(EXTRACT_DIR, 'LA')

if os.path.exists(LA_DIR):
    print(f'✅ Dataset already extracted')
else:
    print(f'Extracting dataset...')
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q {ZIP_PATH} -d {EXTRACT_DIR}
    print(f'✅ Extraction complete')

from models  import MobileStudentCNN, Wav2VecTeacher
from dataset import AudioDeepfakeDataset
from utils   import load_asvspoof2019, compute_eer, kd_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n✅ Setup complete | Device: {device}")
print(f"  Dataset root : {config.DATASET_ROOT}")
print(f"  Save dir     : {config.SAVE_DIR}")

Mounted at /content/drive
Cloning into '/content/project'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 23 (delta 2), reused 22 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 349.47 KiB | 13.44 MiB/s, done.
Resolving deltas: 100% (2/2), done.
Extracting dataset...
✅ Extraction complete

✅ Setup complete | Device: cuda
  Dataset root : /content/asvspoof2019/LA
  Save dir     : /content/drive/MyDrive/deepfake_detector/models


In [ ]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

## 3.1 — Load Sweep Data + Teacher

Load 1,000 balanced samples for sweep training and 200 for validation.
Teacher is loaded once and shared across all 9 experiments.

In [ ]:
# Load sweep data
print("Loading sweep data...")
train_files, train_labels = load_asvspoof2019(
    config.DATASET_ROOT, config.AUDIO_DIRS, config.PROTOCOL_FILES,
    subset="train", max_samples=1000, balanced=True
)
val_files, val_labels = load_asvspoof2019(
    config.DATASET_ROOT, config.AUDIO_DIRS, config.PROTOCOL_FILES,
    subset="dev", max_samples=200, balanced=True
)

assert train_labels.count(0) == train_labels.count(1), "FAIL: Train not balanced!"
assert val_labels.count(0)   == val_labels.count(1),   "FAIL: Val not balanced!"
print(f"✅ Class balance verified")

train_dataset = AudioDeepfakeDataset(train_files, train_labels, is_training=True)
val_dataset   = AudioDeepfakeDataset(val_files,   val_labels,   is_training=False)

train_loader  = DataLoader(train_dataset, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f"  Train batches : {len(train_loader)}")
print(f"  Val batches   : {len(val_loader)}")

# Load teacher once — shared across all experiments
print("\nLoading Teacher (frozen)...")
teacher = Wav2VecTeacher().to(device)
teacher.eval()
print("✅ Teacher loaded and frozen")

**Note:** The output of this cell (Teacher model loading) has been cleared
before pushing to GitHub. The Hugging Face model download progress bar
generates widget metadata that causes GitHub's notebook renderer to show
"Invalid Notebook". The Teacher loads correctly when you run this cell —
it downloads `facebook/wav2vec2-base` (~380MB) on first run and uses
cache on subsequent runs.

## 3.2 — Hyperparameter Sweep

Run 9 combinations × 5 epochs × 1,000 samples.
Goal: find the best Learning Rate, Temperature, and Alpha before committing to full training.

| Parameter | Values tested |
|-----------|--------------|
| Learning Rate | 0.001, 0.0005, 0.0001 |
| Temperature | 4.0, 6.0 |
| Alpha | 0.5, 0.7 |

In [ ]:
def run_experiment(lr, temperature, alpha, train_loader, val_loader, num_epochs=5):
    """Train student for num_epochs, return best val_loss and val_acc."""
    student      = MobileStudentCNN().to(device)
    optimizer    = optim.Adam(student.parameters(), lr=lr)
    scheduler    = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    hard_loss_fn = nn.CrossEntropyLoss()
    soft_loss_fn = nn.KLDivLoss(reduction="batchmean")

    best_val_loss = float('inf')
    best_val_acc  = 0.0

    for epoch in range(1, num_epochs + 1):

        # --- Train ---
        student.train()
        for raw_audio, mel_specs, true_labels in train_loader:
            raw_audio   = raw_audio.to(device)
            mel_specs   = mel_specs.to(device)
            true_labels = true_labels.to(device)

            optimizer.zero_grad()
            with torch.no_grad():
                teacher_logits = teacher(raw_audio)

            student_logits = student(mel_specs)
            loss, _, _     = kd_loss(
                student_logits, teacher_logits, true_labels,
                temperature, alpha, hard_loss_fn, soft_loss_fn
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=1.0)
            optimizer.step()

        scheduler.step()

        # --- Validate ---
        student.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for raw_audio, mel_specs, true_labels in val_loader:
                raw_audio   = raw_audio.to(device)
                mel_specs   = mel_specs.to(device)
                true_labels = true_labels.to(device)

                teacher_logits = teacher(raw_audio)
                student_logits = student(mel_specs)
                loss, _, _     = kd_loss(
                    student_logits, teacher_logits, true_labels,
                    temperature, alpha, hard_loss_fn, soft_loss_fn
                )
                val_loss    += loss.item()
                predicted    = torch.argmax(student_logits, dim=1)
                val_correct += (predicted == true_labels).sum().item()
                val_total   += true_labels.size(0)

        avg_val_loss = val_loss / len(val_loader)
        avg_val_acc  = 100.0 * val_correct / val_total

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_val_acc  = avg_val_acc

    return best_val_loss, best_val_acc


# ==========================================
# Sweep Grid — 9 combinations
# ==========================================
sweep_grid = config.SWEEP_GRID

print(f"{'='*65}")
print(f"  Hyperparameter Sweep — {len(sweep_grid)} combinations × 5 epochs")
print(f"  Train: 1,000 samples | Val: 200 samples")
print(f"{'='*65}")
print(f"  {'LR':<8} {'Temp':<6} {'Alpha':<6} {'Val Loss':<12} {'Val Acc':<10}")
print(f"  {'-'*55}")

sweep_results = []

for i, (lr, temp, alpha) in enumerate(sweep_grid):
    print(f"  Running {i+1}/{len(sweep_grid)}: LR={lr}, T={temp}, Alpha={alpha} ...", end=" ", flush=True)
    val_loss, val_acc = run_experiment(lr, temp, alpha, train_loader, val_loader, num_epochs=5)
    sweep_results.append((lr, temp, alpha, val_loss, val_acc))
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.1f}%")

# Sort by val_loss
sweep_results.sort(key=lambda x: x[3])

print(f"\n{'='*65}")
print(f"  SWEEP RESULTS — sorted by Val Loss (lower is better)")
print(f"{'='*65}")
print(f"  {'Rank':<6} {'LR':<8} {'Temp':<6} {'Alpha':<6} {'Val Loss':<12} {'Val Acc':<10}")
print(f"  {'-'*55}")
for rank, (lr, temp, alpha, val_loss, val_acc) in enumerate(sweep_results, 1):
    marker = " ← BEST" if rank == 1 else ""
    print(f"  {rank:<6} {lr:<8} {temp:<6} {alpha:<6} {val_loss:<12.4f} {val_acc:<10.1f}%{marker}")

# Save best params
best_lr    = sweep_results[0][0]
best_temp  = sweep_results[0][1]
best_alpha = sweep_results[0][2]

print(f"\n✅ Best Parameters Found:")
print(f"   LR          : {best_lr}")
print(f"   Temperature : {best_temp}")
print(f"   Alpha       : {best_alpha}")

  Hyperparameter Sweep — 9 combinations × 5 epochs
  Train: 1,000 samples | Val: 200 samples
  LR       Temp   Alpha  Val Loss     Val Acc   
  -------------------------------------------------------
  Running 1/9: LR=0.001, T=4.0, Alpha=0.7 ... Val Loss: 0.1955 | Val Acc: 69.5%
  Running 2/9: LR=0.001, T=6.0, Alpha=0.7 ... Val Loss: 0.1946 | Val Acc: 67.5%
  Running 3/9: LR=0.001, T=4.0, Alpha=0.5 ... Val Loss: 0.2958 | Val Acc: 79.5%
  Running 4/9: LR=0.0005, T=4.0, Alpha=0.7 ... Val Loss: 0.1865 | Val Acc: 81.5%
  Running 5/9: LR=0.0005, T=6.0, Alpha=0.7 ... Val Loss: 0.1944 | Val Acc: 70.0%
  Running 6/9: LR=0.0005, T=4.0, Alpha=0.5 ... Val Loss: 0.2684 | Val Acc: 89.0%
  Running 7/9: LR=0.0001, T=4.0, Alpha=0.7 ... Val Loss: 0.1929 | Val Acc: 71.5%
  Running 8/9: LR=0.0001, T=6.0, Alpha=0.7 ... Val Loss: 0.1900 | Val Acc: 77.5%
  Running 9/9: LR=0.0001, T=4.0, Alpha=0.5 ... Val Loss: 0.2973 | Val Acc: 79.5%

  SWEEP RESULTS — sorted by Val Loss (lower is better)
  Rank   LR       